# Phase 6: Stronger Ranking Models and Failure Reduction

In [5]:
import sys
! {sys.executable} -m pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------------- ------------------- 0.7/1.5 MB 16.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 18.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score
import lightgbm as lgb
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED=PROJECT_ROOT / 'data' / 'processed'
PHASE5_OUTPUT=DATA_PROCESSED / 'phase5_failure_analysis'
PHASE6_OUTPUT=DATA_PROCESSED / 'phase6_models'
PHASE6_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Phase 6 output: {PHASE6_OUTPUT}")

PIPELINES=['raw', 'global', 'per_query']
K_VALUES=[1, 3, 5, 10]
RANDOM_SEED=36
MAX_PAIRS_PER_QUERY=1000

Phase 6 output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models


## 1. Data Loading

In [7]:
print("="*80)
print("DATA LOADING")
print("="*80)

def load_letor_fold(dataset_name, fold_num):
    base_path=DATA_RAW / dataset_name / f'Fold{fold_num}'
    splits={}
    
    for split_name in ['train', 'vali', 'test']:
        file_path=base_path / f'{split_name}.txt'
        if not file_path.exists():
            raise RuntimeError(f"Missing: {file_path}")
        
        data=[]
        with open(file_path, 'r') as f:
            for line in f:
                if '#' in line:
                    line=line[:line.index('#')]
                parts=line.strip().split()
                if len(parts)<2:
                    continue
                
                label=int(parts[0])
                qid=int(parts[1].split(':')[1])
                features={}
                for item in parts[2:]:
                    if ':' in item:
                        fid, fval=item.split(':')
                        fid=int(fid)
                        if 1<=fid<=46:
                            features[f'f{fid}']=float(fval)
                row={'qid': qid, 'label': label}
                row.update(features)
                data.append(row)
        
        df=pd.DataFrame(data)
        for i in range(1, 47):
            if f'f{i}' not in df.columns:
                df[f'f{i}']=0.0
        df=df.drop(columns=[f'f{i}' for i in range(6, 11)], errors='ignore')
        splits[split_name]=df
        print(f"{dataset_name} {split_name:5s}: {len(df):6d} docs, {df['qid'].nunique():4d} queries")
    
    return splits['train'], splits['vali'], splits['test']

print("\nMQ2007:")
train_2007, vali_2007, test_2007=load_letor_fold('MQ2007', 1)
print("\nMQ2008:")
_, _, test_2008=load_letor_fold('MQ2008', 1)

feature_cols=[c for c in train_2007.columns if c.startswith('f')]
print(f"\nFeatures: {len(feature_cols)}")
print("DATA LOADED")

DATA LOADING

MQ2007:
MQ2007 train:  42158 docs, 1017 queries
MQ2007 vali :  13813 docs,  339 queries
MQ2007 test :  13652 docs,  336 queries

MQ2008:
MQ2008 train:   9630 docs,  471 queries
MQ2008 vali :   2707 docs,  157 queries
MQ2008 test :   2874 docs,  156 queries

Features: 41
DATA LOADED


## 2. Metrics